## Streaming processing of the QUAX experiment data with Dask and Kafka

***Note**: This notebook presents an overview of the architecture and the final benchmarks of our real time ETL pipeline for the QUAX experiment.*

*For the sake of brevity and readability, the complete source code is not included here. Please refer directly to the .py and .ipynb files for the implementation details.*

## Overview

In this project, we build a **real time ETL (Extract-Transform-Load) pipeline for streaming data** produced by the DAQ of the **QUAX experiment**, a resonant RF cavity used in axion dark matter searches. The DAQ produces scans consisting of the In-phase (I) and Quadrature (Q) components, which are two sinusoid functions in quadrature phase and together form a complex-valued time-domain signal (I + jQ). Processing this raw data consists of performing a **Fourier transform** on each scan to move from the time domain to the frequency domain, and then averaging all scans within a data-taking run to obtain a single averaged power spectrum. Each scan contains $2^{11} = 2048$ samples, and the FFT uses the same number of bins, while spanning a $2$ MHz bandwidth.

The dataset is composed of 2 sets of .dat binary files, one for the I measures and the other for the Q, each one comprised of a continuous series of ADC readings from the amplifier. Each ADC reading is written in the raw files as a 32-bit floating point value in little-endian format. Each file contains $2^{23}$ samples and its size is 32 MB. A realistic data production has a **throughput of 16 MB/s** and produces 1 file-pair every about 5 seconds. 

## Architecture

The overall pipeline exploits **Dask** and **Apache Kafka** to perform the full real-time analysis. It consists of four main blocks:
- **Producer**: reads the raw I/Q `.dat` files and emulates a continuous DAQ stream, publishing batches of samples to Kafka topic `topic_stream` with a configurable number of samples per batch and a configurable target throughput. It is hosted in a CloudVeneto virtual machine (VM) for reasons that will be explained later, but can also be hosted on a laptop.
- **Dask cluster**: consumes the raw batches from `topic_stream`, performs the FFT-based processing, and publishes the resulting power spectra to `topic_results`. It is deployed in CloudVeneto, and it hosts 1 scheduler node and up to 4 worker nodes.
- **Kafka broker**: acts as the decoupling layer between data management and processing, buffering the raw stream on `topic_stream` and the processed results on `topic_results`. It is hosted in the same CloudVeneto VM where the Dask scheduler sits.
- **Consumer**: subscribes to `topic_results` and renders a live-updating dashboard, showing both the per-worker spectra and the cumulative averaged spectrum. It is hosted on a local laptop.

Kafka sits between the Producer and the Dask cluster, and again between the Dask cluster and the Consumer, so that each stage can run independently and at its own pace: a slowdown in processing does not block data ingestion, and a slowdown in visualization does not block processing. This decoupling also makes the pipeline horizontally scalable, since Dask workers can be added to increase processing throughput without changing the Producer or Consumer.

We set up Kafka `topic_stream` with a partition number equals to or larger than the number of workers. Indeed, **each worker hosts a Kafka consumer and a Kafka producer**, and we bypass the data streaming through the Dask scheduler which can be a potential bottleneck. This set up also allows the creation of a consumer group of worker nodes that process data in parallel, favouring the construction of an easy scalable system.

The communications between our laptops and the VMs in CloudVeneto are hold via an ssh tunnel. We rely on port forwarding, which allows to connect our laptops with CloudVeneto VMs that are not directly exposed to the internet. All client interactions with the Kafka cluster were implemented in our Python scripts using the kafka-python API (version 3.0.8).

Each CloudVeneto VM has 2 vCPUs and 4GB of RAM (*flavor: cloudveneto-medium*, OS: Debian 13).

![architecture](architecture.png)

### Kafka Broker Settings

The Kafka broker is hosted on the same CloudVeneto VM as the Dask scheduler. Below are the key configuration choices made to ensure the system operates smoothly, all of which are defined within the `kafka-server.properties` file used to start the Kafka broker:

* **Dual Network Listeners (`INTERNAL` / `EXTERNAL`)**: The broker exposes two separate communication channels on different ports. The `INTERNAL` listener (port 19092) is bound to the VM's private IP address (10.67.22.90) and handles all traffic within the CloudVeneto network, including communications with the Dask workers and the Producer. The `EXTERNAL` listener (port 9092) is advertised as `localhost`, allowing us to securely interact with the cluster from our personal laptops via an SSH tunnel.
* **Offset Replication Factor (`offsets.topic.replication.factor=1`)**: This parameter was set to 1, instead of the default value of 3, because we are not using the replication feature.
* **Data Retention Policies (`log.retention.*`)**: The log retention parameters were specifically tailored to the scale and duration of our tests. The minimum log size to retain was capped at approximately 192 MB (structured in 65 MB segments), while the time expiration was set to 10 minutes. Furthermore, we reduced the retention check frequency (`log.retention.check.interval.ms`) from the default 5 minutes to just 10 seconds. This responsiveness is necessary because the standard interval would not trigger the cleanup in time, leading to uncontrolled RAM usage growth and out-of-memory errors.
* **Large Payload Handling (`message.max.bytes`, `socket.*.bytes`)**: The size limits for individual messages and network buffers (both receive and send) were raised to 100 MB. This capacity ensures the system to process the heaviest data batches, which can reach up to 64 MB in size plus headers.

We managed Kafka within the python scripts using kafka-python (version 3.0.8) API.

```properties



# All settings are documented at https://kafka.apache.org/43/configuration/broker-configs/


# A comma-separated list of the names of the listeners used by the controller.
controller.listener.names=CONTROLLER

# The node id associated with this instance's roles
node.id=1

# The role of this server. Setting this puts us in KRaft mode
process.roles=broker,controller

# Listener name, hostname and port the broker or the controller will advertise to clients. INTERNAL refers to cloud VMs and EXTERNAL to our laptops
advertised.listeners=INTERNAL://10.67.22.90:19092,EXTERNAL://localhost:9092

# Disable the background thread checks the distribution of partition leaders at regular intervals. Not needed with a single broker
auto.leader.rebalance.enable=false

# List of controller endpoints used connect to the controller cluster
controller.quorum.bootstrap.servers=localhost:9093

# The address the socket server listens on
listeners=INTERNAL://0.0.0.0:19092,EXTERNAL://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093

# A comma separated list of directories under which to store log files
log.dirs=/tmp/kraft-combined-logs

log.retention.check.interval.ms = 10000

# The minimum age of a log file to be eligible for deletion due to age
log.retention.minutes=10

# A size-based retention policy for logs. Functions independently of log.retention.hours. We set this to the size of 6 files (192MB)
log.retention.bytes=201326592

# The maximum size of a log segment file. When this size is reached a new log segment will be created. We set this to 65MB
log.segment.bytes=68157440

# The largest record batch size allowed by Kafka. We set this to 100MB
message.max.bytes=104857600

# The receive buffer (SO_RCVBUF) used by the socket server
socket.receive.buffer.bytes=104857600

# The maximum size of a request that the socket server will accept (protection against OOM)
socket.request.max.bytes=104857600

# The send buffer (SO_SNDBUF) used by the socket server
socket.send.buffer.bytes=1048576

# Name of listener used for communication between brokers.
inter.broker.listener.name=INTERNAL

# Maps listener names to security protocols.
listener.security.protocol.map=CONTROLLER:PLAINTEXT,INTERNAL:PLAINTEXT,EXTERNAL:PLAINTEXT

# The replication factor for the offsets topic.
offsets.topic.replication.factor=1

### Network Throughput Analysis

We are interested in a general **network throughput analysis** since our architecture completely relies on network communications. Specifically, the rate can be bottlenecked by the underlying network infrastructure. To evaluate this, we measured the throughput using `iperf3`, a bash command that evaluates the throughput between two communicating machines. 

Here, we compare the network throughput between a local laptop and a CloudVeneto VM against the internal throughput between two CloudVeneto nodes. This analysis was crucial for determining the optimal deployment location for the producer. We have found the following results:

*   **Throughput between a local laptop and a CloudVeneto VM:** When measuring through an SSH tunnel, the throughput ranges between 8 MB/s and 25 MB/s. This performance is highly dependent on the local Internet connection and exhibits significant variability even within the same network, making it difficult to guarantee a constant target throughput of 16 MB/s. We corroborated these findings using `speedtest-cli`, a command that measures the internet bandwidth, yielding upload speeds between 12 MB/s and 16 MB/s on our laptop, and slightly higher download speeds on CloudVeneto. While these latter measurements do not account for the SSH tunnel overhead, they provide a reliable baseline estimate.
*   **Throughput between CloudVeneto nodes:** The internal throughput consistently ranges between 375 MB/s and 500 MB/s, representing a highly stable and capable network link. In this context, using `speedtest-cli` is not meaningful, as it measures the connection outside the cluster. 

Based on these findings, **we deployed the producer directly within the CloudVeneto infrastructure** rather than on a local machine, successfully ensuring an ideal 16 MB/s target throughput. Moreover, note that this network measurement is an upper limit of the real throughput achieved by the Producer, which depends on other factors (see later).

Below, we report an excerpt of the terminal script with the tests performed on a local laptop connected via SSH to a CloudVeneto VM:

```bash
dghezzi@PcDaniele:~/Project_MAPD$ iperf3 -c localhost -p 15201
Connecting to host localhost, port 15201
[  5] local 127.0.0.1 port 44338 connected to 127.0.0.1 port 15201
[ ID] Interval           Transfer     Bitrate         Retr  Cwnd
[  5]   0.00-1.02   sec  26.1 MBytes   215 Mbits/sec    5   1.19 MBytes
[  5]   1.02-2.02   sec  18.4 MBytes   154 Mbits/sec    3   1.19 MBytes
[  5]   2.02-3.00   sec  19.9 MBytes   170 Mbits/sec    3   1.19 MBytes
[  5]   3.00-4.01   sec  19.8 MBytes   165 Mbits/sec    5   1.19 MBytes
[  5]   4.01-5.01   sec  21.2 MBytes   178 Mbits/sec    4   1.19 MBytes
[  5]   5.01-6.00   sec  18.2 MBytes   153 Mbits/sec    3   1.19 MBytes
[  5]   6.00-7.00   sec  20.2 MBytes   170 Mbits/sec    6   1.19 MBytes
[  5]   7.00-8.00   sec  19.6 MBytes   165 Mbits/sec    5   1.19 MBytes
[  5]   8.00-9.00   sec  18.2 MBytes   153 Mbits/sec    4   1.19 MBytes
[  5]   9.00-10.05  sec  22.1 MBytes   178 Mbits/sec    5   1.19 MBytes
- - - - - - - - - - - - - - - - - - - - - - - - -
[ ID] Interval           Transfer     Bitrate         Retr
[  5]   0.00-10.05  sec   205 MBytes   171 Mbits/sec   43             sender
[  5]   0.00-9.69   sec   199 MBytes   172 Mbits/sec                  receiver

dghezzi@PcDaniele:~/Project_MAPD$ iperf3 -c localhost -p 15201
Connecting to host localhost, port 15201
[  5] local 127.0.0.1 port 60206 connected to 127.0.0.1 port 15201
[ ID] Interval           Transfer     Bitrate         Retr  Cwnd
[  5]   0.00-1.01   sec  24.2 MBytes   202 Mbits/sec    3   3.68 MBytes
[  5]   1.01-2.00   sec  14.2 MBytes   120 Mbits/sec    1   3.68 MBytes
[  5]   2.00-3.01   sec  13.0 MBytes   109 Mbits/sec    0   3.68 MBytes
[  5]   3.01-4.00   sec  9.50 MBytes  80.1 Mbits/sec    3   3.68 MBytes
[  5]   4.00-5.00   sec  13.6 MBytes   114 Mbits/sec    0   3.68 MBytes
[  5]   5.00-6.01   sec  12.9 MBytes   108 Mbits/sec    3   3.68 MBytes
[  5]   6.01-7.00   sec  13.2 MBytes   112 Mbits/sec    1   3.68 MBytes
[  5]   7.00-8.00   sec  11.5 MBytes  96.4 Mbits/sec    0   3.68 MBytes
[  5]   8.00-9.01   sec  13.6 MBytes   114 Mbits/sec    2   3.68 MBytes
[  5]   9.01-10.01  sec  13.2 MBytes   111 Mbits/sec    3   3.68 MBytes
- - - - - - - - - - - - - - - - - - - - - - - - -
[ ID] Interval           Transfer     Bitrate         Retr
[  5]   0.00-10.01  sec   139 MBytes   117 Mbits/sec   16             sender
[  5]   0.00-9.76   sec   131 MBytes   113 Mbits/sec                  receiver

dghezzi@PcDaniele:~/Project_MAPD$ speedtest-cli
Retrieving speedtest.net configuration...
Testing from Telecom Italia Business (79.1.175.151)...
Retrieving speedtest.net server list...
Selecting best server based on ping...
Hosted by Wolnet Srl (Verona) [71.61 km]: 25.428 ms
Testing download speed...................................................
Download: 122.60 Mbit/s
Testing upload speed.....................................................
Upload: 109.69 Mbit/s

Here we report the connection tests between two distinct CloudVeneto VM:

```bash
debian@mapd-group11-4:~$ iperf3 -c 10.67.22.90
Connecting to host 10.67.22.90, port 5201
[  5] local 10.67.22.236 port 60000 connected to 10.67.22.90 port 5201
[ ID] Interval           Transfer     Bitrate         Retr  Cwnd
[  5]   0.00-1.00   sec   468 MBytes  3.92 Gbits/sec  1039   1.54 MBytes
[  5]   1.00-2.00   sec   479 MBytes  4.02 Gbits/sec    0   1.74 MBytes
[  5]   2.00-3.00   sec   388 MBytes  3.25 Gbits/sec    0   1.84 MBytes
[  5]   3.00-4.00   sec   467 MBytes  3.92 Gbits/sec   17   1.48 MBytes
[  5]   4.00-5.00   sec   442 MBytes  3.71 Gbits/sec    0   1.67 MBytes
[  5]   5.00-6.00   sec   372 MBytes  3.12 Gbits/sec    0   1.76 MBytes
[  5]   6.00-7.00   sec   358 MBytes  3.00 Gbits/sec    0   1.85 MBytes
[  5]   7.00-8.00   sec   348 MBytes  2.91 Gbits/sec    0   1.91 MBytes
[  5]   8.00-9.00   sec   389 MBytes  3.26 Gbits/sec    0   2.04 MBytes
[  5]   9.00-10.00  sec   441 MBytes  3.70 Gbits/sec   64   1.63 MBytes
- - - - - - - - - - - - - - - - - - - - - - - - -
[ ID] Interval           Transfer     Bitrate         Retr
[  5]   0.00-10.00  sec  4.05 GBytes  3.48 Gbits/sec  1120            sender
[  5]   0.00-10.00  sec  4.05 GBytes  3.48 Gbits/sec                  receiver

### Worker Analysis

Since we are interested in the worker nodes' hardware performance, we executed the `lscpu` command on the nodes. We found these specifications, summarized in the following table:

| Worker | Vendor ID | Model Name | CPU(s) | Thread(s) per core | 
| :--- | :--- | :--- | :--- | :--- | 
| **Worker 1** (10.67.22.58) | GenuineIntel | Intel(R) Xeon(R) Gold 6238R CPU @ 2.20GHz | 2 | 1 | 
| **Worker 2** (10.67.22.244) | AuthenticAMD | AMD EPYC 7413 24-Core Processor | 2 | 1 | 
| **Worker 3** (10.67.22.210) | GenuineIntel | Intel(R) Xeon(R) Gold 6238R CPU @ 2.20GHz | 2 | 1 | 
| **Worker 4** (10.67.22.57) | GenuineIntel | Intel(R) Xeon(R) Gold 6238R CPU @ 2.20GHz | 2 | 1 | 

### Producer

The **Producer emulates the QUAX DAQ** by reading I/Q raw data and streaming it to Kafka as if it were being acquired live. At the start, it loads the full set of `duck_i_*.dat`/`duck_q_*.dat` file pairs into memory as **raw byte buffers**, avoiding repeated disk I/O during the streaming loop.

The stream is then played in fixed size batches, controlled by two configurable parameters:
* **`N_SCANS_PER_BATCH`**: the number of scans grouped into a single Kafka message.
* **`TARGET_THROUGHPUT_MB`**: the target data rate (in MB/s) the Producer tries to sustain, matching the real DAQ's rate of 16 MB/s.

For each batch, the **I and Q byte slices are concatenated into a single datum** and published to `topic_stream`: thus we can replicate exactly the DAQ, which sends a pair of I and Q files every 5 seconds, by setting `N_SCANS_PER_BATCH` = 4096. The Producer also sends dedicated **control signals** (`BEGIN_STREAM` and `END_STREAM`): the `BEGIN_STREAM` signal transmits configuration metadata (target throughput, scans per batch, ...) to initialize the downstream processing, while the `END_STREAM` signal marks the conclusion of the data transmission, asking the workers to publish their final accumulated results. In between, the **Producer calls `send()` and `flush()`**: the first function appends the message in a memory buffer and returns a future object, while the second function pushes all the buffered messages to the broker (and waits for their acknowledgment). This logic allows us to implement control over the pacing of each batch, necessary to reliably fix `TARGET_THROUGHPUT_MB` rather than letting Kafka's internal batching logic determine it implicitly.

The Producer is run from a dedicated VM inside CloudVeneto (connecting directly to the `INTERNAL` listener).

### Dask Cluster 

The **Dask cluster** is deployed in CloudVeneto and consists of one scheduler node and up to 4 worker nodes, each running on a VM with 2 vCPUs and 4GB of RAM. The scheduler is located on the same VM as the Kafka broker. Unlike a typical Dask workload, the scheduler leverages the **Dask Actors** by instantiating a stateful `WorkerProcessor` class on each worker via `dask_client.submit(..., actor=True)`. In Dask, an Actor is a Python object located on a worker that maintains its internal state, such as network connections and data accumulators. Beyond launching these actors, the scheduler is not involved in the data path at all: **each worker owns its own Kafka consumer and producer, so the raw stream and the results stream both bypass the scheduler entirely and flow directly between the workers and the broker**. This avoids turning the scheduler into a throughput bottleneck and keeps the design close to a Kafka consumer-group pattern rather than a classic Dask task graph.

Each worker's consumer joins the same **consumer group**, allowing parallel processing of partitioned data. For every batch pulled from `topic_stream`, the worker reshapes the complex I/Q samples into a `(n_scans, N_SAMPLES_PER_SCAN)` matrix, allowing vectorized single-batch processing: it computes a row-wise FFT to obtain the power spectrum of each scan, then evaluates the mean and second moment across all scans. Rather than storing every scan, the worker maintains a running mean and a running sum of squared deviations of the power spectrum, updated using [Welford's algorithm](https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Welford's_online_algorithm): this saves memory and keeps its usage constant regardless of how long the stream runs.

Since we aim to refresh the Consumer dashboard every 5 seconds, we set `publish_interval_sec` accordingly: once this interval has elapsed, the worker publishes the accumulated mean, second moment (M2), and scan count together with batch metadata (producer timestamps, processing times, waiting times) to `topic_results`, then resets its state for the next time interval. On receiving the producer's `END_STREAM` signal, each worker flushes any remaining partial accumulation and forwards an `END_STREAM` message downstream, so the dashboard can detect when all workers have finished.

### Dashboard

The **dashboard is a live matplotlib dashboard** that subscribes to `topic_results` and runs locally on our laptops. Messages are fetched via `consumer.poll(timeout_ms=100)` inside the main event loop, allowing the application to process incoming data while staying responsive. To maintain a clean architecture, the logic is decoupled into dedicated classes: `WorkerState`, `GlobalState`, `Dashboard`, and `BenchmarkLogger`. 

For each message, the `WorkerState` updates its individual metrics, and then the `GlobalState` merges this partial spectrum into a single cumulative global spectrum using Welford's algorithm. This lets the `Dashboard` show both the individual worker spectra and one combined average spectrum, with global plot error bars derived from the accumulated second moment.

Alongside the plots, the system uses every incoming message to compute network and processing latencies, which are collected by the `BenchmarkLogger` to be written out to a JSON file for later offline analysis. Stream control and exit conditions are managed by handling `BEGIN_STREAM` and `END_STREAM` headers, which track the active status of each worker.

## Benchmark

### Bug and Bottlenecks Fixing  

During the development of the pipeline, we encountered three main problems:

* **Producer Stalls (`kafka-python` Bug):** While using the `kafka-python` library (version 3.0.7), we traced a Producer internal bug (an `IndexError` while encoding message headers). This bug silently killed the background sender thread, which explained the instances where messages were never delivered to the worker nodes. Upgrading the library to version 3.0.8 definitively resolved the issue.

* **The Python Checksum (CRC32C) Issue:** Apache Kafka ensures data integrity by computing a CRC32C checksum for every batch of messages sent or received. Initially, we noticed a very low throughput in the Producer, making it unable to sustain the 16 MB/s rate. The issue stemmed from the `kafka-python` library, which by default uses a pure Python implementation to compute the checksum that is extremely slow. Installing the dedicated C module on the Producer via `pip install crc32c` ([see here](https://pypi.org/project/kafka-python/#description)) removed this computational bottleneck, allowing us to sustain the target throughput of 16 MB/s.

* **Network issues/Broker crash:** For unknown reasons, sometimes the producer disconnects from Kafka broker. Sometime because the Kafka broker crashes due to running out of memory